# Pre-Training Verification Checklist

**Status:** ✓ All A100 optimizations implemented and ready to train

In [1]:
print("Hello NeurIPS")

Hello NeurIPS


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Neur — Sequential Colab Pipeline

Run each cell in order. Every step reads from and writes to Google Drive so you can stop and resume between steps.

- **Step 0** — clone repo, install deps, mount Drive.
- **Step 1** — download Samanantar (2M pairs) + FLORES-200 to `Drive/neur/raw_data/`.
- **Step 2** — clean and split the raw data into `Drive/neur/processed_data/`.
- **Step 3** — train the sinusoidal model and evaluate on FLORES-200. Outputs in `Drive/neur/outputs/`.


In [11]:
%cd /content
!git clone https://github.com/thenileshmishra/AS-RoPE.git neur
%cd /content/neur
!git pull

/content
Cloning into 'neur'...
remote: Enumerating objects: 293, done.
remote: Counting objects: 100% (293/293), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 293 (delta 143), reused 235 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (293/293), 1.08 MiB | 10.52 MiB/s, done.
Resolving deltas: 100% (143/143), done.
/content/neur
Already up to date.


In [13]:
!ls

notebooks  pipeline  README.md	requirements.txt  src


In [17]:
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.53 KiB | 782.00 KiB/s, done.
From https://github.com/thenileshmishra/AS-RoPE
   27ca197..7f02c90  main       -> origin/main
Updating 27ca197..7f02c90
Fast-forward
 notebooks/pipeline.ipynb | 78 ++++++++++++++++++++----------------------------
 pipeline/step3_train.py  |  6 +++-
 2 files changed, 38 insertions(+), 46 deletions(-)


In [14]:
import os, sys
os.environ['NEUR_DRIVE_ROOT'] = '/content/drive/MyDrive/neur'
sys.path.insert(0, '/content/neur')

from pipeline import paths
paths.ensure_dirs()
print(paths.summary())

PROJECT_ROOT    = /content/drive/MyDrive/neur
RAW_SAMANANTAR  = /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
RAW_FLORES      = /content/drive/MyDrive/neur/raw_data/flores200/flores200_hi_en_devtest.tsv
PROCESSED_DIR   = /content/drive/MyDrive/neur/processed_data
CHECKPOINT_DIR  = /content/drive/MyDrive/neur/outputs/checkpoints
LOGS_DIR        = /content/drive/MyDrive/neur/outputs/logs
METRICS_DIR     = /content/drive/MyDrive/neur/outputs/metrics


## Step 1 — Download raw datasets to Google Drive

Downloads up to 2M Samanantar pairs and the full FLORES-200 devtest split.
Safe to re-run — existing files are skipped unless `--force` is passed.

In [17]:
!python -m pipeline.step1_download --force

[step1] paths:
PROJECT_ROOT    = /content/drive/MyDrive/neur
RAW_SAMANANTAR  = /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
RAW_FLORES      = /content/drive/MyDrive/neur/raw_data/flores200/flores200_hi_en_devtest.tsv
PROCESSED_DIR   = /content/drive/MyDrive/neur/processed_data
CHECKPOINT_DIR  = /content/drive/MyDrive/neur/outputs/checkpoints
LOGS_DIR        = /content/drive/MyDrive/neur/outputs/logs
METRICS_DIR     = /content/drive/MyDrive/neur/outputs/metrics
[step1] loading ai4bharat/samanantar/hi ...
[step1] samanantar total rows: 10,125,706
[step1] using all 10,125,706 rows
[step1] detected hi=tgt en=src translation_dict=False
[step1] progress: 99,999 pairs written...
[step1] progress: 199,999 pairs written...
[step1] progress: 299,999 pairs written...
[step1] progress: 399,999 pairs written...
[step1] progress: 499,999 pairs written...
[step1] progress: 599,999 pairs written...
[step1] progress: 699,999 pairs written...
[step1] progress: 799,999 pairs writt

## Step 2 — Clean and split the raw data

Reads `raw_data/samanantar/samanantar_hi_en.tsv`, applies cleaning filters
(length, length-ratio, script, dedupe), and writes deterministic train/val/test
splits to `processed_data/`.

In [ ]:
!python -m pipeline.step2_preprocess

[step2] streaming raw pairs from /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
2026-04-11 13:25:13.802969: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775913914.546110   14898 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775913914.661792   14898 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775913915.929431   14898 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775913915.929489   14898 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target m

In [ ]:
!ls -lh /content/drive/MyDrive/neur/outputs/checkpoints
!cat /content/drive/MyDrive/neur/outputs/metrics/metrics.json

## Step 3 — Train (sinusoidal) + evaluate on FLORES-200

Trains the GPT model with sinusoidal positional encoding, saving checkpoints under `outputs/checkpoints/`, training logs under `outputs/logs/`, and BLEU/chrF under `outputs/metrics/`.

In [ ]:
!python -m pipeline.step3_train
--pe-type sinusoidal
--use-bf16
--pin-memory --dataloader-workers 2
--grad-accum 4 --batch-size 64
--max-seq-len 128
--num-steps 20000
--eval-every 2000



[step3] device=cuda (NVIDIA A100-SXM4-40GB, 42.4 GB)
[step3] config: pe_type=sinusoidal d_model=512 n_layers=12 batch=64x8=512 steps=100000 bf16=True compile=True
config.json: 100% 832/832 [00:00<00:00, 4.85MB/s]
tokenizer_config.json: 100% 498/498 [00:00<00:00, 1.47MB/s]
spiece.model:  90% 1.70M/1.90M [00:19<00:00, 8.50MB/s]^C


## Verify outputs on Drive

In [ ]:
!ls -lh /content/drive/MyDrive/neur/outputs/checkpoints
!cat /content/drive/MyDrive/neur/outputs/metrics/metrics.json